# 수묵화 데이터셋 전처리 파이프라인


In [ ]:
# ============================
# 라이브러리 임포트
# ============================
import os
import glob
import json
from pathlib import Path

import numpy as np          # 수치 연산 (배열, 통계 등)
import cv2                  # OpenCV: 이미지 처리 (리사이즈, 엣지검출, 색공간 변환)
from PIL import Image       # 이미지 열기/메타데이터 확인용
import matplotlib.pyplot as plt        # 시각화 (그래프, 이미지 출력)
import matplotlib.font_manager as fm   # 한글 폰트 설정용
from tqdm.notebook import tqdm         # 진행률 표시 바
import pandas as pd         # 데이터프레임으로 이미지 정보/특징값 관리


def imread_bgr(path) -> np.ndarray | None:
    """OpenCV imread는 Windows 한글 경로를 읽지 못하므로 imdecode 사용."""
    data = np.fromfile(str(path), dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, cv2.IMREAD_COLOR)


def imread_gray(path) -> np.ndarray | None:
    data = np.fromfile(str(path), dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)

In [ ]:
# ============================
# 경로 설정
# ============================
# 프로젝트 루트 (notebooks/ 기준 한 단계 위)
PROJECT_ROOT = Path("../")

# AI Hub 원본 데이터 위치 (HYU/BUHT_data)
DATA_ROOT = Path("../../BUHT_data")

# 사용할 데이터 분할: "Train" | "Validation" | "all"
# - Train: 학습용 (약 5,800장)
# - Validation: 검증용 (약 900장)
# - all: Train + Validation 전체
USE_SPLIT = "Train"

if USE_SPLIT == "all":
    RAW_DIR = DATA_ROOT
else:
    RAW_DIR = DATA_ROOT / USE_SPLIT

# 전처리 결과는 BUHT 프로젝트 안에 저장
# data/processed/ → 전처리(리사이즈+필터링) 완료된 이미지
# data/sketches/  → ControlNet 학습용 스케치(엣지) 이미지
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SKETCH_DIR = PROJECT_ROOT / "data" / "sketches"

# 폴더가 없으면 자동 생성
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SKETCH_DIR.mkdir(parents=True, exist_ok=True)

# 이미지로 인식할 확장자 목록
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}

print(f"DATA_ROOT: {DATA_ROOT.resolve()}")
print(f"RAW_DIR:   {RAW_DIR.resolve()}")
print(f"PROCESSED_DIR: {PROCESSED_DIR.resolve()}")
print(f"SKETCH_DIR: {SKETCH_DIR.resolve()}")

## 1. 데이터 로딩 & 기본 탐색

In [ ]:
def collect_image_paths(root_dir: Path) -> list[Path]:
    """
    root_dir 하위의 모든 이미지 파일 경로를 수집한다.
    - 하위 폴더까지 재귀적으로 탐색 (rglob)
    - 대소문자 모두 처리 (.jpg, .JPG 등)
    - 중복 제거 후 정렬하여 반환
    """
    paths = []
    for ext in IMG_EXTENSIONS:
        paths.extend(root_dir.rglob(f"*{ext}"))       # 소문자 확장자
        paths.extend(root_dir.rglob(f"*{ext.upper()}"))  # 대문자 확장자
    return sorted(set(paths))  # 중복 제거 + 정렬


# 원본 이미지 경로 수집
image_paths = collect_image_paths(RAW_DIR)
print(f"총 이미지 수: {len(image_paths)}")

# 이미지가 없으면 안내 메시지 출력
if len(image_paths) == 0:
    print(f"\n[!] {RAW_DIR.resolve()} 에 이미지가 없습니다.")
    print("    경로 설정에서 DATA_ROOT, USE_SPLIT 값을 확인해주세요.")

In [ ]:
def get_image_info(path: Path) -> dict:
    """
    이미지 파일 하나의 기본 정보를 딕셔너리로 반환한다.
    - width, height: 이미지 가로/세로 크기 (픽셀)
    - aspect_ratio: 종횡비 (가로/세로)
    - format: 파일 포맷 (JPEG, PNG 등)
    - mode: 색상 모드 (RGB, L 등)
    - filesize_kb: 파일 크기 (KB)
    열 수 없는 이미지는 error 키에 에러 메시지를 담아 반환
    """
    try:
        img = Image.open(path)
        w, h = img.size
        return {
            "path": str(path),
            "filename": path.name,
            "width": w,
            "height": h,
            "aspect_ratio": round(w / h, 2),
            "format": img.format,
            "mode": img.mode,
            "filesize_kb": round(path.stat().st_size / 1024, 1),
        }
    except Exception as e:
        return {"path": str(path), "error": str(e)}


# 전체 이미지의 기본 정보를 수집하여 DataFrame으로 정리
if image_paths:
    infos = [get_image_info(p) for p in tqdm(image_paths, desc="이미지 정보 수집")]
    df_info = pd.DataFrame(infos)

    # 정상 이미지와 오류(손상/열 수 없는) 이미지를 분리
    if "error" in df_info.columns:
        df_errors = df_info[df_info["error"].notna()].copy()
        df_valid = df_info[df_info["error"].isna()].copy()
    else:
        df_errors = df_info.iloc[0:0].copy()
        df_valid = df_info.copy()

    print(f"\n정상 이미지: {len(df_valid)}")
    print(f"오류 이미지: {len(df_errors)}")
    print(f"\n--- 해상도 통계 (min, mean, max 등) ---")
    print(df_valid[["width", "height", "filesize_kb"]].describe())
else:
    print("이미지가 없으므로 건너뜁니다.")

In [ ]:
def show_sample_grid(paths: list[Path], n=16, figsize=(16, 16)):
    """
    랜덤으로 n장을 뽑아 그리드 형태로 출력한다.
    데이터가 어떤 이미지들로 구성되어 있는지 눈으로 확인하는 용도.
    """
    n = min(n, len(paths))
    indices = np.random.choice(len(paths), n, replace=False)  # 랜덤 인덱스
    cols = int(np.ceil(np.sqrt(n)))   # 그리드 열 수
    rows = int(np.ceil(n / cols))     # 그리드 행 수

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        img = Image.open(paths[idx])
        axes[i].imshow(img)
        axes[i].set_title(paths[idx].name, fontsize=8)
        axes[i].axis("off")

    # 남는 칸은 비워두기
    for i in range(n, len(axes)):
        axes[i].axis("off")

    plt.suptitle("랜덤 샘플 이미지", fontsize=14)
    plt.tight_layout()
    plt.show()


if image_paths:
    show_sample_grid(image_paths)

## 2. 수묵화 특징 검증 필터

수묵화의 핵심 특징을 수치화하여 검증합니다:
- **채도(Saturation)**: 수묵화는 무채색 위주 → 낮은 채도
- **여백 비율**: 밝은 배경 영역이 넓음
- **그레이스케일 유사도**: R, G, B 채널 간 차이가 적음
- **엣지 밀도**: 부드러운 붓터치 → 낮은 엣지 밀도

In [ ]:
def compute_ink_features(img_path: Path) -> dict:
    """
    수묵화 판별을 위한 5가지 특징값을 계산한다.

    [수묵화의 시각적 특징]
    - 먹과 물로만 그리므로 채도가 낮다 (거의 무채색)
    - 여백의 미: 빈 공간(흰 배경)이 많다
    - RGB 채널 값이 거의 동일하다 (그레이스케일에 가까움)
    - 붓터치가 부드러워 엣지(경계선)가 적다

    Returns:
        dict:
        - mean_saturation: HSV 채도 평균 (0~255, 낮을수록 무채색)
        - whitespace_ratio: 밝은 픽셀(V>200) 비율 (0~1, 높을수록 여백 많음)
        - grayscale_similarity: RGB 채널 간 표준편차 평균 (낮을수록 그레이스케일)
        - edge_density: Canny 엣지 픽셀 비율 (0~1, 낮을수록 부드러운 그림)
        - mean_value: HSV 명도 평균 (0~255)
    """
    img_bgr = imread_bgr(img_path)
    if img_bgr is None:
        return None

    # BGR → HSV 변환 (Hue: 색상, Saturation: 채도, Value: 명도)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    # BGR → 그레이스케일 변환 (엣지 검출용)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    h, s, v = cv2.split(img_hsv)  # HSV 각 채널 분리
    b, g, r = cv2.split(img_bgr)  # BGR 각 채널 분리

    # ---- 특징 1: 채도(Saturation) 평균 ----
    # 수묵화는 먹+물이라 채도가 매우 낮음 (보통 0~30 범위)
    mean_saturation = float(np.mean(s))

    # ---- 특징 2: 여백 비율 ----
    # 명도(V)가 200 이상인 픽셀 = 거의 흰색 = 여백
    # 수묵화는 여백의 미가 있어서 이 비율이 높음
    whitespace_ratio = float(np.mean(v > 200))

    # ---- 특징 3: 그레이스케일 유사도 ----
    # 각 픽셀에서 R, G, B 값의 표준편차를 구함
    # 수묵화는 R≈G≈B이므로 std가 매우 낮음
    # 채색화(단청, 민화 등)는 R,G,B 차이가 커서 std가 높음
    rgb_stack = np.stack([r, g, b], axis=-1).astype(np.float32)
    pixel_std = np.std(rgb_stack, axis=-1)
    grayscale_similarity = float(np.mean(pixel_std))

    # ---- 특징 4: 엣지 밀도 ----
    # Canny edge detection으로 경계선 검출
    # 수묵화는 번짐/농담 표현이 많아 엣지가 적음
    # 디지털 일러스트나 만화는 선이 뚜렷해서 엣지가 많음
    edges = cv2.Canny(img_gray, 50, 150)
    edge_density = float(np.mean(edges > 0))

    # ---- 특징 5: 평균 명도 ----
    # 참고용 (직접 필터링에는 안 쓰지만 분포 확인용)
    mean_value = float(np.mean(v))

    return {
        "path": str(img_path),
        "mean_saturation": round(mean_saturation, 2),
        "whitespace_ratio": round(whitespace_ratio, 4),
        "grayscale_similarity": round(grayscale_similarity, 2),
        "edge_density": round(edge_density, 4),
        "mean_value": round(mean_value, 2),
    }

In [ ]:
FEAT_CSV = PROJECT_ROOT / "outputs" / "ink_features.csv"
FORCE_REEXTRACT = False  # True면 CSV 무시하고 다시 추출

if FEAT_CSV.exists() and not FORCE_REEXTRACT:
    df_feat = pd.read_csv(FEAT_CSV)
    print(f"CSV 로드 완료: {len(df_feat)}장  ({FEAT_CSV.resolve()})")
    display(df_feat.describe())
elif image_paths:
    features = []
    for p in tqdm(image_paths, desc="수묵화 특징 추출"):
        feat = compute_ink_features(p)
        if feat is not None:
            features.append(feat)

    df_feat = pd.DataFrame(features)
    print(f"특징 추출 완료: {len(df_feat)}장")
    if len(df_feat) == 0:
        print("[!] 특징을 추출한 이미지가 없습니다.")
    else:
        display(df_feat.describe())
else:
    print("이미지가 없으므로 건너뜁니다.")


In [ ]:
# df_feat CSV 저장
if "df_feat" in globals() and not df_feat.empty:
    out_dir = PROJECT_ROOT / "outputs"
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / "ink_features.csv"
    df_feat.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {csv_path.resolve()}")
else:
    print("저장할 df_feat가 없습니다.")

In [ ]:
def plot_feature_distributions(df: pd.DataFrame):
    """
    5가지 특징값의 히스토그램을 그린다.
    빨간 점선 = 중앙값(median)
    → 이 분포를 보고 THRESHOLDS 임계값을 조정하면 됨
    """
    feature_cols = ["mean_saturation", "whitespace_ratio", "grayscale_similarity", "edge_density", "mean_value"]
    titles = ["채도 평균", "여백 비율", "그레이스케일 유사도\n(낮을수록 무채색)", "엣지 밀도", "평균 명도"]

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, col, title in zip(axes, feature_cols, titles):
        ax.hist(df[col], bins=50, edgecolor="black", alpha=0.7)
        ax.set_title(title)
        # 중앙값을 빨간 점선으로 표시
        ax.axvline(df[col].median(), color="red", linestyle="--", label=f"median={df[col].median():.2f}")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


if image_paths and not df_feat.empty:
    plot_feature_distributions(df_feat)

In [ ]:
# ============================
# 수묵화 필터링 임계값 (Thresholds)
# ============================
# [중요] 위 히스토그램을 보고 아래 값을 데이터에 맞게 조정하세요!
# 너무 엄격하면 수묵화도 걸러지고, 너무 느슨하면 비수묵화가 포함됩니다.

THRESHOLDS = {
    # 채도 평균 상한: 이 값 이하 = 무채색(수묵화)
    # 수묵화는 보통 0~30, 채색화는 50 이상
    "max_saturation": 50,

    # 여백 비율 하한: 이 값 이상 = 여백이 충분함
    # 수묵화는 보통 0.2~0.7, 꽉 찬 그림은 0.05 이하
    "min_whitespace_ratio": 0.1,

    # 그레이스케일 유사도 상한: 이 값 이하 = R≈G≈B (무채색)
    # 수묵화는 보통 0~10, 채색화는 20 이상
    "max_grayscale_sim": 15,

    # 엣지 밀도 상한: 이 값 이하 = 부드러운 붓터치
    # 수묵화는 보통 0.02~0.10, 만화/일러스트는 0.15 이상
    "max_edge_density": 0.15,
}


def is_ink_painting(row: pd.Series, thresholds: dict = THRESHOLDS) -> bool:
    """
    4가지 조건을 모두 만족하면 수묵화로 판정 (AND 조건).
    하나라도 벗어나면 비수묵화로 분류됨.
    """
    return (
        row["mean_saturation"] <= thresholds["max_saturation"]
        and row["whitespace_ratio"] >= thresholds["min_whitespace_ratio"]
        and row["grayscale_similarity"] <= thresholds["max_grayscale_sim"]
        and row["edge_density"] <= thresholds["max_edge_density"]
    )


# 각 이미지에 수묵화 여부 판정 적용
if image_paths and not df_feat.empty:
    df_feat["is_ink"] = df_feat.apply(is_ink_painting, axis=1)
    n_ink = df_feat["is_ink"].sum()
    n_total = len(df_feat)
    print(f"수묵화 판정: {n_ink} / {n_total} ({n_ink/n_total*100:.1f}%)")
    print(f"비수묵화 (제거 대상): {n_total - n_ink}장")

In [ ]:
def show_filtered_comparison(df: pd.DataFrame, n=8):
    """
    수묵화로 판정된 이미지(윗줄)와 제외된 이미지(아랫줄)를 비교 출력한다.
    → 필터링이 제대로 작동하는지 눈으로 확인하는 용도
    → 비수묵화 쪽에 수묵화가 섞여 있으면 THRESHOLDS 값을 조정해야 함
    """
    ink_paths = df[df["is_ink"]]["path"].tolist()
    non_ink_paths = df[~df["is_ink"]]["path"].tolist()

    fig, axes = plt.subplots(2, n, figsize=(n * 2.5, 6))

    # 윗줄: 수묵화 판정 (O 표시)
    for i in range(n):
        if i < len(ink_paths):
            img = Image.open(ink_paths[i])
            axes[0][i].imshow(img)
            axes[0][i].set_title("O", fontsize=10, color="green")
        axes[0][i].axis("off")
    axes[0][0].set_ylabel("수묵화", fontsize=12)

    # 아랫줄: 비수묵화 판정 (X 표시)
    for i in range(n):
        if i < len(non_ink_paths):
            img = Image.open(non_ink_paths[i])
            axes[1][i].imshow(img)
            axes[1][i].set_title("X", fontsize=10, color="red")
        axes[1][i].axis("off")
    axes[1][0].set_ylabel("비수묵화", fontsize=12)

    plt.suptitle("수묵화 필터링 결과 비교", fontsize=14)
    plt.tight_layout()
    plt.show()


if image_paths and not df_feat.empty:
    show_filtered_comparison(df_feat)

## 3. 전처리 적용

In [ ]:
# ============================
# 전처리 설정
# ============================
TARGET_SIZE = 512   # Stable Diffusion 입력 크기 기준 (512x512)
MIN_RESOLUTION = 256  # 가로 또는 세로가 256px 미만이면 품질 부족으로 제외


def preprocess_image(img_path: str, target_size: int = TARGET_SIZE) -> np.ndarray | None:
    """
    이미지 한 장을 전처리한다.

    처리 과정:
    1. 저해상도(256px 미만) 이미지는 제외 (None 반환)
    2. 긴 변 기준으로 target_size에 맞게 비율 유지 리사이즈
    3. 남는 영역은 흰색(255)으로 패딩 → 정사각형 출력
       (수묵화 배경이 흰색이므로 패딩이 자연스러움)

    Args:
        img_path: 이미지 파일 경로
        target_size: 출력 이미지 크기 (가로=세로)

    Returns:
        전처리된 numpy 배열 (target_size x target_size x 3) 또는 None
    """
    img = imread_bgr(img_path)
    if img is None:
        return None

    h, w = img.shape[:2]

    # 너무 작은 이미지는 학습에 부적합하므로 제외
    if min(h, w) < MIN_RESOLUTION:
        return None

    # 비율 유지 리사이즈: 긴 변을 target_size에 맞춤
    scale = target_size / max(h, w)
    new_w = int(w * scale)
    new_h = int(h * scale)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # 흰색(255) 캔버스에 중앙 배치 → 정사각형 만들기
    canvas = np.full((target_size, target_size, 3), 255, dtype=np.uint8)
    y_offset = (target_size - new_h) // 2  # 상하 여백
    x_offset = (target_size - new_w) // 2  # 좌우 여백
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized

    return canvas

In [ ]:
# 수묵화로 판정된 이미지만 전처리하여 data/processed/에 저장
if not df_feat.empty:
    ink_df = df_feat[df_feat["is_ink"]].copy()
    ink_image_paths = ink_df["path"].tolist()

    processed_count = 0
    skipped_count = 0

    for img_path in tqdm(ink_image_paths, desc="전처리 중"):
        result = preprocess_image(img_path)
        if result is not None:
            # 원본 파일명 유지, 확장자만 png로 통일
            filename = Path(img_path).stem + ".png"
            save_path = PROCESSED_DIR / filename
            cv2.imwrite(str(save_path), result)
            processed_count += 1
        else:
            skipped_count += 1

    print(f"\n전처리 완료: {processed_count}장 저장")
    print(f"저해상도 제외: {skipped_count}장")
    print(f"저장 경로: {PROCESSED_DIR.resolve()}")
else:
    print("이미지가 없으므로 건너뜁니다.")

## 4. ControlNet용 스케치 추출

Canny Edge Detection으로 스케치 라인을 추출합니다.  
나중에 HED 등 더 정교한 방법으로 교체할 수 있습니다.

In [ ]:
def extract_sketch(img_path: str, low_thresh: int = 30, high_thresh: int = 100) -> np.ndarray | None:
    """
    Canny Edge Detection으로 스케치(라인 드로잉)를 추출한다.

    ControlNet의 Canny 모델에 입력할 스케치 이미지를 생성하는 함수.
    수묵화의 윤곽선만 추출하여 흰 배경 + 검은 라인 형태로 출력.

    처리 과정:
    1. 그레이스케일로 읽기
    2. 가우시안 블러로 노이즈 제거 (불필요한 잡선 방지)
    3. Canny edge detection으로 윤곽선 검출
    4. 반전 (Canny 출력: 검은 배경+흰 라인 → 흰 배경+검은 라인)

    Args:
        img_path: 이미지 파일 경로
        low_thresh: Canny 하한 임계값 (낮을수록 더 많은 엣지 검출)
        high_thresh: Canny 상한 임계값 (높을수록 강한 엣지만 검출)
    """
    img = imread_gray(img_path)
    if img is None:
        return None

    # 가우시안 블러: 5x5 커널로 노이즈 제거
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    # Canny edge detection
    edges = cv2.Canny(blurred, low_thresh, high_thresh)

    # 반전: 흰 배경 + 검은 라인 (ControlNet Canny 입력 형식)
    sketch = 255 - edges
    return sketch

In [ ]:
# 전처리 완료된 이미지(data/processed/)에서 스케치를 추출하여 data/sketches/에 저장
# 파일명은 원본과 동일하게 유지 (1:1 매칭)
processed_paths = sorted(PROCESSED_DIR.glob("*.png"))
if processed_paths:

    sketch_count = 0
    for p in tqdm(processed_paths, desc="스케치 추출 중"):
        sketch = extract_sketch(str(p))
        if sketch is not None:
            save_path = SKETCH_DIR / p.name  # 동일 파일명으로 저장
            cv2.imwrite(str(save_path), sketch)
            sketch_count += 1

    print(f"스케치 추출 완료: {sketch_count}장")
    print(f"저장 경로: {SKETCH_DIR.resolve()}")
else:
    print("이미지가 없으므로 건너뜁니다.")

In [ ]:
def show_original_vs_sketch(processed_dir: Path, sketch_dir: Path, n=4):
    """
    전처리된 원본(윗줄)과 추출된 스케치(아랫줄)를 나란히 비교한다.
    → 스케치 품질이 괜찮은지 눈으로 확인하는 용도
    → 라인이 너무 적거나 많으면 extract_sketch()의 low_thresh, high_thresh 조정
    """
    processed_paths = sorted(processed_dir.glob("*.png"))[:n]
    if not processed_paths:
        print(f"비교할 이미지가 없습니다: {processed_dir.resolve()}")
        return

    n = len(processed_paths)
    fig, axes = plt.subplots(2, n, figsize=(n * 3, 6))
    if n == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for i, p in enumerate(processed_paths):
        img = imread_bgr(p)
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[0][i].imshow(img_rgb)
        axes[0][i].set_title(p.name, fontsize=8)
        axes[0][i].axis("off")

        sketch_path = sketch_dir / p.name
        sketch = imread_gray(sketch_path)
        if sketch is not None:
            axes[1][i].imshow(sketch, cmap="gray")
        axes[1][i].axis("off")

    axes[0][0].set_ylabel("원본", fontsize=12)
    axes[1][0].set_ylabel("스케치", fontsize=12)
    plt.suptitle("원본 vs 스케치 비교", fontsize=14)
    plt.tight_layout()
    plt.show()


processed_paths = sorted(PROCESSED_DIR.glob("*.png"))
if processed_paths:
    show_original_vs_sketch(PROCESSED_DIR, SKETCH_DIR)
else:
    print(f"전처리 이미지가 없습니다. 전처리 저장을 먼저 실행하세요.")

## 5. 메타데이터 & 통계 리포트

In [ ]:
# ============================
# 최종 결과 요약 & CSV 리포트 저장
# ============================
if not df_feat.empty:
    n_raw = len(image_paths) if "image_paths" in globals() else len(df_feat)
    print("=" * 50)
    print("        전처리 결과 요약")
    print("=" * 50)
    print(f"원본 이미지 수:        {n_raw}")
    print(f"수묵화 판정:           {df_feat['is_ink'].sum()}")
    print(f"비수묵화 제외:         {(~df_feat['is_ink']).sum()}")
    print(f"전처리 저장:           {len(list(PROCESSED_DIR.glob('*.png')))}")
    print(f"스케치 추출:           {len(list(SKETCH_DIR.glob('*.png')))}")
    print("=" * 50)

    # 각 이미지의 특징값 + 수묵화 판정 결과를 CSV로 저장
    # → 팀원과 공유하거나, 나중에 임계값 재조정할 때 활용
    report_dir = PROJECT_ROOT / "outputs"
    report_dir.mkdir(parents=True, exist_ok=True)
    report_path = report_dir / "preprocessing_report.csv"
    df_feat.to_csv(report_path, index=False)
    print(f"\n특징값 리포트 저장: {report_path.resolve()}")
else:
    print(f"데이터가 없습니다. {RAW_DIR.resolve()} 경로를 확인한 후 처음부터 다시 실행하세요.")

## 6. Caption 생성 (SDXL 학습용 테스트)

파이프라인의 **Text Prompt T**를 자동 생성하기 위한 VLM caption 테스트입니다.

- 입력: `data/processed/` 수묵화 이미지
- 출력: 이미지 생성 학습에 최적화된 영문 caption
- 소수 샘플로 프롬프트 품질 확인 후 전체 확장

**아래 설정 셀에 Luxia Bridge API 정보를 입력하세요.** (`Agent/final_code.py`와 동일한 방식: `API_KEY` + `BRIDGE_URL`)

In [ ]:
# ============================
# Caption 생성 설정 (Luxia Bridge)
# ============================
import base64
import json
import random
import re
import time
from pathlib import Path

import requests

# 상단 (앞 셀 미실행 시에만 보정)
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path("..").resolve()
if "PROCESSED_DIR" not in globals():
    PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# --- API 설정 (Agent/final_code.py 와 동일) ---
API_KEY = ""  # Luxia Bridge apikey
BRIDGE_URL = "https://bridge.luxiacloud.com/llm/openai/chat/completions/gpt-4o/create"
MODEL = "gpt-4o-2024-08-06"
HEADERS = {"apikey": API_KEY, "Content-Type": "application/json"}

# --- 테스트 설정 ---
CAPTION_TEST_N = 3       # 테스트할 이미지 수
RANDOM_SEED = 42
TRIGGER_WORD = "traditional ink wash painting"   # 전통 먹 화풍 (LoRA 트리거, caption 끝에 붙임)

CAPTION_DIR = PROJECT_ROOT / "outputs" / "captions"
CAPTION_DIR.mkdir(parents=True, exist_ok=True)
CAPTION_TEST_PATH = CAPTION_DIR / "caption_test.json"

# SDXL + ControlNet + IP-Adapter 학습에 맞춘 시스템 프롬프트
CAPTION_SYSTEM_PROMPT = """You caption images for training Stable Diffusion XL on East Asian ink wash painting (sumukhwa / suiboku / sumi-e).

Write captions optimized for generative model training, not for human gallery descriptions.

Rules:
- English only, 1-2 sentences, 35-70 words
- Lead with subject + composition (what is painted, where placed, viewpoint)
- Describe ink technique: brush strokes, ink density, wash gradients, negative space
- Note sparse color washes only if visible; default to monochrome ink
- Include mood/atmosphere
- Avoid: "image of", "photo", camera/lens terms, "masterpiece", "8k", artist names
- Do NOT invent objects not clearly visible

Return ONLY valid JSON:
{
  "caption": "<full training caption>",
  "subject": "<main subject, few words>",
  "composition": "<brief layout>",
  "style_tags": ["tag1", "tag2", "tag3"]
}"""

print(f"MODEL: {MODEL}")
print(f"BRIDGE_URL: {BRIDGE_URL}")
print(f"API_KEY set: {bool(API_KEY)}")
print(f"Test images: {CAPTION_TEST_N}")
print(f"Output: {CAPTION_TEST_PATH.resolve()}")


In [ ]:
def image_to_data_url(path: Path) -> str:
    suffix = path.suffix.lower().lstrip(".")
    mime = "jpeg" if suffix in {"jpg", "jpeg"} else suffix
    b64 = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:image/{mime};base64,{b64}"


def parse_json_response(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if m:
            return json.loads(m.group(0))
        raise


def build_training_caption(parsed: dict, trigger_word: str = TRIGGER_WORD) -> str:
    base = parsed["caption"].strip().rstrip(".")
    tags = parsed.get("style_tags", [])
    tag_str = ", ".join(tags[:4]) if tags else "ink wash painting, sumi-e"
    return f"{base}, {tag_str}, {trigger_word}"


def _post_chat(messages, timeout=90, max_retries=3) -> str:
    """Luxia Bridge chat-completions 호출 (retry + backoff)."""
    for attempt in range(max_retries):
        try:
            payload = {
                "model": MODEL,
                "messages": messages,
                "stream": False,
                "temperature": 0.2,
                "top_p": 1.0,
                "max_tokens": 400,
            }
            r = requests.post(BRIDGE_URL, headers=HEADERS, json=payload, timeout=timeout)
            if r.status_code != 200:
                if attempt < max_retries - 1:
                    time.sleep(0.5 * (2 ** attempt))
                    continue
                raise RuntimeError(f"status={r.status_code}, body={r.text[:300]}")
            return r.json()["choices"][0]["message"]["content"].strip()
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            if attempt < max_retries - 1:
                time.sleep(0.5 * (2 ** attempt) + random.uniform(0, 0.2))
                continue
            raise


def caption_image_with_vlm(image_path: Path, model: str = MODEL, trigger_word: str = TRIGGER_WORD) -> dict:
    data_url = image_to_data_url(image_path)
    raw = _post_chat([
        {"role": "system", "content": CAPTION_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Caption this ink wash painting for SDXL fine-tuning. Focus on visible content and brushwork."},
                {"type": "image_url", "image_url": {"url": data_url}},
            ],
        },
    ])
    parsed = parse_json_response(raw)
    parsed["training_caption"] = build_training_caption(parsed, trigger_word)
    parsed["raw_response"] = raw
    return parsed

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

if "imread_bgr" not in globals():
    def imread_bgr(path) -> np.ndarray | None:
        data = np.fromfile(str(path), dtype=np.uint8)
        if data.size == 0:
            return None
        return cv2.imdecode(data, cv2.IMREAD_COLOR)

rng = np.random.default_rng(RANDOM_SEED)
all_processed = sorted(PROCESSED_DIR.glob("*.png"))

if not all_processed:
    raise FileNotFoundError(f"전처리 이미지가 없습니다: {PROCESSED_DIR.resolve()}")

n_sample = min(CAPTION_TEST_N, len(all_processed))
sample_indices = rng.choice(len(all_processed), size=n_sample, replace=False)
caption_test_paths = [all_processed[i] for i in sorted(sample_indices)]

print(f"선택된 테스트 이미지 {len(caption_test_paths)}장:")
for p in caption_test_paths:
    print(" -", p.name)

fig, axes = plt.subplots(1, len(caption_test_paths), figsize=(3 * len(caption_test_paths), 3))
if len(caption_test_paths) == 1:
    axes = [axes]
for ax, p in zip(axes, caption_test_paths):
    img = imread_bgr(p)
    if img is not None:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(p.stem[:20], fontsize=7)
    ax.axis("off")
plt.suptitle("Caption 테스트 샘플", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
if not API_KEY:
    print("[!] API_KEY가 비어 있습니다. 위 설정 셀에 Luxia Bridge apikey를 입력하세요.")
else:
    caption_results = []

    for path in tqdm(caption_test_paths, desc="Caption 생성"):
        try:
            result = caption_image_with_vlm(path)
            record = {
                "filename": path.name,
                "image_path": str(path.resolve()),
                "subject": result.get("subject", ""),
                "composition": result.get("composition", ""),
                "caption": result.get("caption", ""),
                "style_tags": result.get("style_tags", []),
                "training_caption": result.get("training_caption", ""),
            }
        except Exception as e:
            record = {"filename": path.name, "image_path": str(path.resolve()), "error": str(e)}
        caption_results.append(record)
        time.sleep(0.3)

    with open(CAPTION_TEST_PATH, "w", encoding="utf-8") as f:
        json.dump(caption_results, f, ensure_ascii=False, indent=2)

    ok = sum(1 for r in caption_results if "error" not in r)
    print(f"\n완료: {ok}/{len(caption_results)} 성공")
    print(f"저장: {CAPTION_TEST_PATH.resolve()}")

In [ ]:
# requests는 대부분 환경에 기본 설치됨. 없으면 아래 주석 해제
# %pip install requests

In [ ]:
if CAPTION_TEST_PATH.exists():
    with open(CAPTION_TEST_PATH, encoding="utf-8") as f:
        caption_results = json.load(f)

    valid = [r for r in caption_results if "training_caption" in r and "error" not in r]
    errors = [r for r in caption_results if "error" in r]

    if errors:
        print(f"Failed: {len(errors)}/{len(caption_results)}")
        for r in errors:
            print(f"  - {r['filename']}: {r['error']}")

    if valid:
        n = len(valid)
        print(f"Caption test results ({n} images)\n")

        for i, r in enumerate(valid, start=1):
            img = imread_bgr(Path(r["image_path"]))
            if img is not None:
                plt.figure(figsize=(5, 5))
                plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                plt.title(r["filename"], fontsize=9)
                plt.axis("off")
                plt.tight_layout()
                plt.show()

            print(f"[{i}/{n}] {r['filename']}")
            print(f"Subject          : {r.get('subject', '')}")
            print(f"Composition      : {r.get('composition', '')}")
            print(f"Caption          : {r.get('caption', '')}")
            print(f"Training caption : {r.get('training_caption', '')}")
            print("-" * 60)
    elif not errors:
        print("No caption results found.")
else:
    print("caption_test.json not found. Run the caption generation cell first.")


## 7. Batch Caption Generation (all processed images)

Images sharing the same base ID are treated as one source image with different edits.
Example: `img_1_1_0001(1232)_Scan_edit` and `img_1_1_0001(1234)_Scan_edit` → base ID `img_1_1_0001`.

- One API call per base ID
- Same `training_caption` copied to every variant as a `.txt` file next to each image
- Manifest saved to `outputs/captions/captions_all.json`

**Prerequisite:** Run cells 25–26 first (API settings + helper functions).

In [ ]:
# ============================
# Batch caption generation
# ============================
import re
from collections import defaultdict

if "caption_image_with_vlm" not in globals():
    raise RuntimeError("Run cells 25–26 first (API settings + helper functions).")

PROCESSED_DIR = Path(r"C:\Users\msyu7\Desktop\HYU\BUHT\data\processed")
CAPTION_MANIFEST_PATH = CAPTION_DIR / "captions_all.json"
SKIP_EXISTING = True   # skip groups whose .txt files already exist
SLEEP_SEC = 0.3


def extract_base_id(path: Path) -> str:
    """img_1_1_0001(1232)_Scan_edit -> img_1_1_0001"""
    stem = path.stem
    m = re.match(r"^(.+?)\(\d+\)", stem)
    return m.group(1) if m else stem


def group_processed_images(image_dir: Path) -> dict[str, list[Path]]:
    groups: dict[str, list[Path]] = defaultdict(list)
    for p in sorted(image_dir.glob("*.png")):
        groups[extract_base_id(p)].append(p)
    return dict(groups)


def caption_txt_path(image_path: Path) -> Path:
    return image_path.with_suffix(".txt")


def group_is_complete(paths: list[Path]) -> bool:
    return all(caption_txt_path(p).exists() and caption_txt_path(p).stat().st_size > 0 for p in paths)


def write_caption_files(paths: list[Path], training_caption: str) -> None:
    for p in paths:
        caption_txt_path(p).write_text(training_caption, encoding="utf-8")


def load_manifest() -> dict:
    if CAPTION_MANIFEST_PATH.exists():
        with open(CAPTION_MANIFEST_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {"groups": {}}


def save_manifest(manifest: dict) -> None:
    with open(CAPTION_MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)


if not API_KEY:
    raise ValueError("API_KEY is empty. Set it in cell 25.")

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(f"Processed image folder not found: {PROCESSED_DIR}")

groups = group_processed_images(PROCESSED_DIR)
all_images = sorted(PROCESSED_DIR.glob("*.png"))
manifest = load_manifest()

print(f"Processed dir : {PROCESSED_DIR.resolve()}")
print(f"Total images  : {len(all_images)}")
print(f"Unique bases  : {len(groups)}")
print(f"Manifest      : {CAPTION_MANIFEST_PATH.resolve()}")

api_calls = 0
skipped = 0
failed = 0
written_files = 0

for base_id, paths in tqdm(sorted(groups.items()), desc="Batch captions"):
    paths = sorted(paths)
    representative = paths[0]

    if SKIP_EXISTING and group_is_complete(paths):
        skipped += 1
        continue

    try:
        result = caption_image_with_vlm(representative)
        training_caption = result["training_caption"]
        api_calls += 1
    except Exception as e:
        failed += 1
        manifest["groups"][base_id] = {
            "representative": representative.name,
            "variants": [p.name for p in paths],
            "error": str(e),
        }
        save_manifest(manifest)
        time.sleep(SLEEP_SEC)
        continue

    write_caption_files(paths, training_caption)
    written_files += len(paths)

    manifest["groups"][base_id] = {
        "representative": representative.name,
        "variants": [p.name for p in paths],
        "subject": result.get("subject", ""),
        "composition": result.get("composition", ""),
        "caption": result.get("caption", ""),
        "style_tags": result.get("style_tags", []),
        "training_caption": training_caption,
    }
    save_manifest(manifest)
    time.sleep(SLEEP_SEC)

print("\nDone.")
print(f"API calls     : {api_calls}")
print(f"Skipped groups: {skipped}")
print(f"Failed groups : {failed}")
print(f"Caption files : {written_files}")
print(f"Manifest saved: {CAPTION_MANIFEST_PATH.resolve()}")
